# Kaggle GPU OCR — EasyOCR(CRAFT) detector + VietOCR recognizer

Runs on a **free Kaggle T4 GPU** (~10x faster than local CPU). Produces OCR cache
parquets that drop straight into our local pipeline (same schema as `run_ocr.py`).

## Setup on Kaggle (do this first)
1. New Notebook **inside the competition** (so the data mounts automatically).
2. Right sidebar → **Accelerator = GPU T4 x2** (or any GPU).
3. Right sidebar → **Internet = ON** (needed to pip-install + download model weights).
4. **Run Cell 1 (installs).** Then **Run → Restart & clear cell outputs** (the install
   swaps Pillow; a restart is required so the fixed Pillow loads cleanly).
5. After restart, **Run All**. Cell 1 re-runs fast from cache.
6. When done, download from the right sidebar **Output**:
   `ocr_vietocr_test.parquet` and `ocr_vietocr_all.parquet`.
7. Locally, drop both files into `URA_Hackathon_Kaggle_Competition_2026/cache/`.

Why this stack: VietOCR is a Vietnamese-native recognizer (handles diacritics far
better than PaddleOCR's Latin `vi` model). It's recognition-only, so we use EasyOCR's
CRAFT detector for boxes. Both run on GPU via torch (no PaddlePaddle needed).

> **Troubleshooting:** if Cell 4 ever shows `ImportError: cannot import name
> 'is_directory' from 'PIL._util'`, the Pillow pin didn't take — re-run Cell 1, then
> **restart the kernel** and Run All.

In [ ]:
# Install OCR deps. easyocr/vietocr corrupt Kaggle's pre-installed Pillow
# (ImageFont references PIL._util.is_directory, removed in newer Pillow) -> ImportError.
# Fix: force ONE consistent Pillow AFTER installing the OCR packages.
!pip install -q easyocr vietocr
!pip install -q --force-reinstall "Pillow==10.4.0"
import PIL, torch
print('Pillow', PIL.__version__, '| CUDA', torch.cuda.is_available(),
      '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))

In [ ]:
import os, re, zipfile, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
from tqdm.auto import tqdm

# ---- locate competition data ----
KIN = Path('/kaggle/input')
ROOT = None
for p in KIN.rglob('test.csv'):
    ROOT = p.parent; break
assert ROOT is not None, 'test.csv not found under /kaggle/input — add the competition data'
WORK = Path('/kaggle/working')
print('ROOT =', ROOT)

def _ensure_dir(cands, zips, out_name):
    for rel in cands:
        d = ROOT / rel
        if d.is_dir() and any(d.glob('*.jpg')):
            return d
    for zp in zips:
        z = ROOT / zp
        if z.is_file():
            ex = WORK / out_name
            if not (ex.exists() and any(ex.rglob('*.jpg'))):
                ex.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(z) as zf: zf.extractall(ex)
            for d in [ex, *[c for c in ex.rglob('*') if c.is_dir()]]:
                if any(d.glob('*.jpg')): return d
    raise FileNotFoundError(f'images not found for {out_name}')

TEST_DIR = _ensure_dir(['test_images/images','images','test_images'],
                       ['images.zip','test_images.zip'], 'test_images')
TRAIN_DIR = _ensure_dir(['train_images/train_images','train_images'],
                        ['train_images.zip'], 'train_images')
print('TEST_DIR =', TEST_DIR, '|', len(list(TEST_DIR.glob('*.jpg'))), 'jpg')
print('TRAIN_DIR=', TRAIN_DIR, '|', len(list(TRAIN_DIR.glob('*.jpg'))), 'jpg')
test_ids = pd.read_csv(ROOT/'test.csv')['image_id'].tolist()
train_ids = pd.read_csv(ROOT/'train.csv')['image_id'].tolist()
print('test', len(test_ids), '| train', len(train_ids))

In [ ]:
# ---- post-processing (mirrors local ocr_postprocess.clean_ocr) ----
MAX_OCR_LEN = 500
def normalize_text(t):
    if not t: return ''
    t = unicodedata.normalize('NFC', str(t)).replace('\n',' ').replace('\t',' ').replace('\r',' ')
    return re.sub(r'\s+',' ',t).strip()
def dedupe_consecutive(t):
    toks = t.split()
    if not toks: return ''
    out=[toks[0]]
    for w in toks[1:]:
        if w.lower()!=out[-1].lower(): out.append(w)
    return ' '.join(out)
def clean_ocr(t):
    t = dedupe_consecutive(normalize_text(t))
    return t[:MAX_OCR_LEN].rstrip() if len(t)>MAX_OCR_LEN else t

def crop_box(img, box):
    """box = [x_min,x_max,y_min,y_max] (easyocr horizontal). Returns PIL crop."""
    x0,x1,y0,y1 = box
    h,w = img.shape[:2]
    x0,x1 = max(0,int(x0)), min(w,int(x1)); y0,y1 = max(0,int(y0)), min(h,int(y1))
    if x1<=x0 or y1<=y0: return None
    return Image.fromarray(cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_BGR2RGB))

In [ ]:
import easyocr
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

detector = easyocr.Reader(['vi','en'], gpu=True, verbose=False)  # CRAFT detector
cfg = Cfg.load_config_from_name('vgg_transformer')
cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['predictor']['beamsearch'] = False
recognizer = Predictor(cfg)
print('detector + VietOCR (vgg_transformer) ready on', cfg['device'])

In [ ]:
def ocr_image(path):
    img = cv2.imread(str(path))
    if img is None: return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    try:
        horiz, _free = detector.detect(img, min_size=15, text_threshold=0.6)
        boxes = horiz[0] if horiz else []
    except Exception:
        boxes = []
    items = []  # (y, x, crop)
    for b in boxes:
        crop = crop_box(img, b)
        if crop is not None and crop.size[0] >= 3 and crop.size[1] >= 3:
            items.append((b[2], b[0], crop))
    if not items:
        return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    # batch recognize for speed
    try:
        texts = recognizer.predict_batch([it[2] for it in items])
    except Exception:
        texts = [recognizer.predict(it[2]) for it in items]
    # reading order: top->bottom, left->right with row banding
    ys = [it[0] for it in items]
    band = max(8.0, (max(ys)-min(ys))/40.0)
    order = sorted(range(len(items)), key=lambda i: (round(items[i][0]/band), items[i][1]))
    ordered = [texts[i] for i in order if str(texts[i]).strip()]
    text = clean_ocr(' '.join(ordered))
    raw = clean_ocr(' '.join(str(t) for t in texts))
    return {'ocr_text':text,'raw_text':raw,'mean_conf':0.0,'n_boxes':len(items),'n_chars':len(text)}

def run(ids, img_dir, out_path, save_every=200):
    done = {}
    if Path(out_path).exists():
        prev = pd.read_parquet(out_path); done = {r['image_id']:r for r in prev.to_dict('records')}
    recs = list(done.values())
    pending = [i for i in ids if i not in done]
    print(f'{out_path}: pending {len(pending)}/{len(ids)}')
    for k,iid in enumerate(tqdm(pending)):
        r = ocr_image(img_dir/f'{iid}.jpg'); r['image_id']=iid; recs.append(r)
        if (k+1)%save_every==0: pd.DataFrame(recs).to_parquet(out_path, index=False)
    df = pd.DataFrame(recs); df.to_parquet(out_path, index=False)
    print('saved', out_path, '| fill', (df.ocr_text.str.strip()!='').mean())
    return df

In [ ]:
# TEST first (smaller — 2006), then TRAIN (4892). Each saves to /kaggle/working.
run(test_ids, TEST_DIR, '/kaggle/working/ocr_vietocr_test.parquet')
run(train_ids, TRAIN_DIR, '/kaggle/working/ocr_vietocr_all.parquet')
print('\nDONE — download both parquets from the Output panel.')